# CinnaAI — Additional evaluation cells
Copy these cells, in order, to the end of the original training notebook after Phase 23.
They depend on that notebook’s runtime variables and exported models. This is not a standalone training notebook.
See COLAB_GRAPH_AND_METRIC_CELLS.md for explanations and optional input settings.

## Additional evaluation cell 1


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import sklearn
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, precision_recall_curve,
    average_precision_score, mean_absolute_error,
    mean_squared_error, mean_absolute_percentage_error, r2_score,
)

if "MODEL_DRIVE_PATH" not in globals():
    raise RuntimeError("Run the original Drive/path setup cells first.")
if "test_dataset" not in globals():
    raise RuntimeError("Recreate the ORIGINAL held-out test_dataset first.")

CG_MODEL_DIR = Path(MODEL_DRIVE_PATH)
CG_H5 = Path(globals().get(
    "H5_OUTPUT_PATH", CG_MODEL_DIR / "cinnamon_multi_part_model.h5"
))
CG_CLASS_FILE = Path(globals().get(
    "CN_OUTPUT_PATH", CG_MODEL_DIR / "class_names.json"
))
if not CG_H5.is_file() or not CG_CLASS_FILE.is_file():
    raise FileNotFoundError("Run original Phase 23 export first.")

CG_CLASSES = json.loads(CG_CLASS_FILE.read_text())
CG_K = len(CG_CLASSES)
if CG_K != 7 or len(set(CG_CLASSES)) != CG_K:
    raise ValueError("Expected seven unique output class names.")

CG_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
CG_OUT = CG_MODEL_DIR / "evaluation_reports" / CG_RUN_ID
CG_OUT.mkdir(parents=True, exist_ok=False)

cg_model = tf.keras.models.load_model(str(CG_H5), compile=False)
if tuple(cg_model.input_shape[-3:]) != (224, 224, 3):
    raise ValueError("Exported model input must be 224×224 RGB.")
if cg_model.output_shape[-1] != CG_K:
    raise ValueError("Output count does not match class_names.json.")

def cg_clean_json(value):
    if isinstance(value, dict):
        return {str(k): cg_clean_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [cg_clean_json(v) for v in value]
    if isinstance(value, np.ndarray):
        return cg_clean_json(value.tolist())
    if isinstance(value, np.generic):
        return cg_clean_json(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value

def cg_save_json(name, value):
    (CG_OUT / name).write_text(
        json.dumps(cg_clean_json(value), indent=2, allow_nan=False),
        encoding="utf-8",
    )

def cg_save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(CG_OUT / f"{name}.png", dpi=250, bbox_inches="tight")
    fig.savefig(CG_OUT / f"{name}.pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)

def cg_label_indices(labels):
    values = np.asarray(labels)
    if values.ndim == 2 and values.shape[1] == CG_K:
        if not (
            np.all((values == 0) | (values == 1))
            and np.all(values.sum(axis=1) == 1)
        ):
            raise ValueError("Evaluation needs hard one-hot labels, not MixUp targets.")
        values = np.argmax(values, axis=1)
    else:
        values = values.reshape(-1)
    if not np.all(np.isfinite(values)):
        raise ValueError("Non-finite test labels.")
    if not np.all(values == np.floor(values)):
        raise ValueError("Test labels must be hard class IDs, not MixUp targets.")
    values = values.astype(np.int64)
    if np.any((values < 0) | (values >= CG_K)):
        raise ValueError("Test labels are outside the configured class range.")
    return values

def cg_predict_dataset(candidate):
    true_batches, probability_batches = [], []
    model_seconds = 0.0
    loop_started = time.perf_counter()
    for images, labels in test_dataset:
        started = time.perf_counter()
        probabilities = candidate(images, training=False).numpy()
        model_seconds += time.perf_counter() - started
        targets = cg_label_indices(labels)
        if probabilities.shape != (len(targets), CG_K):
            raise ValueError("Unexpected probability matrix shape.")
        if not np.all(np.isfinite(probabilities)):
            raise ValueError("Non-finite model outputs.")
        if np.any(probabilities < -1e-6) or np.any(probabilities > 1 + 1e-6):
            raise ValueError("Expected probabilities in [0,1].")
        if not np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4):
            raise ValueError("Model outputs do not sum to one.")
        true_batches.append(targets)
        probability_batches.append(probabilities)
    if not true_batches:
        raise ValueError("The test dataset is empty.")
    return (
        np.concatenate(true_batches),
        np.concatenate(probability_batches),
        model_seconds,
        time.perf_counter() - loop_started,
    )

cg_environment = {
    "run_id_utc": CG_RUN_ID,
    "python": platform.python_version(),
    "tensorflow": tf.__version__,
    "numpy": np.__version__,
    "scikit_learn": sklearn.__version__,
    "devices": [str(x) for x in tf.config.list_physical_devices()],
    "class_order": CG_CLASSES,
    "model_path": str(CG_H5),
    "input_contract": "RGB float32, 224x224, values 0-255; no /255",
}
cg_save_json("environment.json", cg_environment)
print("Output folder:", CG_OUT)
print("Class order:", CG_CLASSES)


## Additional evaluation cell 2


In [ ]:
CG_HISTORY_JSON = None  # Optional: "/content/drive/.../training_history.json"
cg_histories = {}

if CG_HISTORY_JSON is not None:
    cg_histories = json.loads(Path(CG_HISTORY_JSON).read_text())
else:
    for cg_stage, cg_variable in [
        ("Initial training", "initial_history"),
        ("Fine-tuning", "finetune_history"),
    ]:
        cg_history_object = globals().get(cg_variable)
        if cg_history_object is not None:
            cg_history_data = getattr(cg_history_object, "history", cg_history_object)
            cg_histories[cg_stage] = {
                key: np.asarray(values, dtype=float).tolist()
                for key, values in cg_history_data.items()
            }

cg_save_json("training_history.json", cg_histories)

def cg_plot_history(metric, title, output_name):
    available = [
        (stage, history) for stage, history in cg_histories.items()
        if len(history.get(metric, [])) > 0
    ]
    if not available:
        print(f"{title}: original training history is unavailable.")
        return
    fig, axes = plt.subplots(
        1, len(available), figsize=(7 * len(available), 5), squeeze=False
    )
    for axis, (stage, history) in zip(axes.ravel(), available):
        values = history[metric]
        axis.plot(range(1, len(values) + 1), values, label="Training")
        validation = history.get(f"val_{metric}", [])
        if len(validation) > 0:
            axis.plot(
                range(1, len(validation) + 1), validation, label="Validation"
            )
        axis.set(title=stage, xlabel="Completed epoch", ylabel=metric.title())
        axis.grid(alpha=0.3)
        axis.legend()
    fig.suptitle(title)
    cg_save_figure(fig, output_name)

cg_plot_history("accuracy", "Training and Validation Accuracy", "training_accuracy")
print("Actual epoch counts:", {
    stage: len(history.get("accuracy", []))
    for stage, history in cg_histories.items()
})


## Additional evaluation cell 3


In [ ]:
cg_plot_history("loss", "Training and Validation Loss", "training_loss")


## Additional evaluation cell 4


In [ ]:
cg_y_true, cg_y_prob, cg_eval_model_seconds, cg_eval_loop_seconds = (
    cg_predict_dataset(cg_model)
)
cg_y_pred = np.argmax(cg_y_prob, axis=1)
cg_one_hot = np.eye(CG_K, dtype=np.int32)[cg_y_true]
cg_support = np.bincount(cg_y_true, minlength=CG_K)

def cg_classification_metrics(true_labels, probabilities):
    predictions = np.argmax(probabilities, axis=1)
    summary = {"accuracy": float(accuracy_score(true_labels, predictions))}
    for average in ["macro", "weighted"]:
        precision, recall, f1, _ = precision_recall_fscore_support(
            true_labels, predictions, labels=np.arange(CG_K),
            average=average, zero_division=0,
        )
        summary[f"precision_{average}"] = float(precision)
        summary[f"recall_{average}"] = float(recall)
        summary[f"f1_{average}"] = float(f1)
    rows = []
    precision, recall, f1, support = precision_recall_fscore_support(
        true_labels, predictions, labels=np.arange(CG_K),
        average=None, zero_division=0,
    )
    valid_aucs, valid_aps = [], []
    for index, label in enumerate(CG_CLASSES):
        binary = (true_labels == index).astype(int)
        has_both = len(np.unique(binary)) == 2
        auc_value = (
            float(roc_auc_score(binary, probabilities[:, index]))
            if has_both else None
        )
        ap_value = (
            float(average_precision_score(binary, probabilities[:, index]))
            if has_both else None
        )
        if has_both:
            valid_aucs.append(auc_value)
            valid_aps.append(ap_value)
        rows.append({
            "class": label, "support": int(support[index]),
            "precision": float(precision[index]),
            "recall": float(recall[index]), "f1": float(f1[index]),
            "roc_auc_ovr": auc_value, "average_precision": ap_value,
        })
    # The full seven-class macro AUC is reported only if every class is defined.
    summary["roc_auc_macro_all_classes"] = (
        float(np.mean(valid_aucs)) if len(valid_aucs) == CG_K else None
    )
    summary["average_precision_macro_all_classes"] = (
        float(np.mean(valid_aps)) if len(valid_aps) == CG_K else None
    )
    summary["auc_defined_class_count"] = len(valid_aucs)
    encoded = np.eye(CG_K, dtype=int)[true_labels]
    summary["roc_auc_micro"] = float(
        roc_auc_score(encoded.ravel(), probabilities.ravel())
    )
    summary["average_precision_micro"] = float(
        average_precision_score(encoded.ravel(), probabilities.ravel())
    )
    summary["test_samples"] = int(len(true_labels))
    return summary, pd.DataFrame(rows)

cg_summary, cg_per_class = cg_classification_metrics(cg_y_true, cg_y_prob)
cg_summary["evaluation_model_call_seconds"] = cg_eval_model_seconds
cg_summary["evaluation_loop_seconds"] = cg_eval_loop_seconds
cg_save_json("classification_metrics.json", cg_summary)
cg_per_class.to_csv(CG_OUT / "per_class_metrics.csv", index=False)

cg_predictions = pd.DataFrame({
    "evaluation_row": np.arange(len(cg_y_true)),
    "actual_class_id": cg_y_true,
    "predicted_class_id": cg_y_pred,
    "actual_class": [CG_CLASSES[index] for index in cg_y_true],
    "predicted_class": [CG_CLASSES[index] for index in cg_y_pred],
    "correct": cg_y_true == cg_y_pred,
    "top_probability": cg_y_prob.max(axis=1),
})
for cg_index, cg_class in enumerate(CG_CLASSES):
    cg_predictions[f"probability_{cg_class}"] = cg_y_prob[:, cg_index]
cg_predictions.to_csv(CG_OUT / "test_predictions.csv", index=False)
np.savez_compressed(
    CG_OUT / "test_arrays.npz",
    y_true=cg_y_true, y_pred=cg_y_pred, probabilities=cg_y_prob,
    class_names=np.asarray(CG_CLASSES),
)
display(pd.DataFrame([cg_summary]).T.rename(columns={0: "value"}))
display(cg_per_class)
if np.any(cg_support == 0):
    print("Missing test classes:", [
        CG_CLASSES[i] for i in np.flatnonzero(cg_support == 0)
    ])
    print("Undefined class AUC/AP are left empty, not converted to zero.")


## Additional evaluation cell 5


In [ ]:
cg_roc_records = []
cg_fig, cg_axis = plt.subplots(figsize=(10, 7))
for cg_index, cg_class in enumerate(CG_CLASSES):
    cg_binary = cg_one_hot[:, cg_index]
    if len(np.unique(cg_binary)) != 2:
        print("Skipping undefined ROC:", cg_class)
        continue
    cg_fpr, cg_tpr, cg_thresholds = roc_curve(
        cg_binary, cg_y_prob[:, cg_index], drop_intermediate=False
    )
    cg_auc = roc_auc_score(cg_binary, cg_y_prob[:, cg_index])
    cg_axis.plot(cg_fpr, cg_tpr, label=f"{cg_class}: AUC={cg_auc:.4f}")
    for fpr, tpr, threshold in zip(cg_fpr, cg_tpr, cg_thresholds):
        cg_roc_records.append({
            "class": cg_class, "fpr": fpr, "tpr": tpr,
            "threshold": threshold,
        })

cg_micro_fpr, cg_micro_tpr, _ = roc_curve(
    cg_one_hot.ravel(), cg_y_prob.ravel()
)
cg_axis.plot(
    cg_micro_fpr, cg_micro_tpr, linestyle=":", linewidth=3,
    label=f"Micro-average: AUC={cg_summary['roc_auc_micro']:.4f}",
)
cg_axis.plot([0, 1], [0, 1], "k--", label="Random ranking")
cg_axis.set(
    xlabel="False positive rate", ylabel="True positive rate (recall)",
    title="Held-out Disease Classification — ROC Curves",
    xlim=(0, 1), ylim=(0, 1.03),
)
cg_axis.legend(loc="lower right", fontsize=9)
cg_axis.grid(alpha=0.25)
pd.DataFrame(cg_roc_records).to_csv(CG_OUT / "roc_points.csv", index=False)
cg_save_figure(cg_fig, "roc_curves")


## Additional evaluation cell 6


In [ ]:
cg_pr_records = []
cg_fig, cg_axis = plt.subplots(figsize=(10, 7))
for cg_index, cg_class in enumerate(CG_CLASSES):
    cg_binary = cg_one_hot[:, cg_index]
    if len(np.unique(cg_binary)) != 2:
        print("Skipping undefined PR/AP:", cg_class)
        continue
    cg_precision, cg_recall, cg_thresholds = precision_recall_curve(
        cg_binary, cg_y_prob[:, cg_index]
    )
    cg_ap = average_precision_score(cg_binary, cg_y_prob[:, cg_index])
    cg_axis.step(
        cg_recall, cg_precision, where="post",
        label=f"{cg_class}: AP={cg_ap:.4f}",
    )
    # PR has one more precision/recall point than threshold values.
    for precision, recall, threshold in zip(
        cg_precision, cg_recall, np.append(cg_thresholds, np.nan)
    ):
        cg_pr_records.append({
            "class": cg_class, "precision": precision,
            "recall": recall, "threshold": threshold,
        })

cg_micro_precision, cg_micro_recall, _ = precision_recall_curve(
    cg_one_hot.ravel(), cg_y_prob.ravel()
)
cg_axis.step(
    cg_micro_recall, cg_micro_precision, where="post",
    linestyle=":", linewidth=3,
    label=f"Micro-average: AP={cg_summary['average_precision_micro']:.4f}",
)
cg_axis.axhline(
    1 / CG_K, color="gray", linestyle="--",
    label="Micro positive-prevalence reference (1/7)",
)
cg_axis.set(
    xlabel="Recall", ylabel="Precision",
    title="Held-out Disease Classification — Precision-Recall",
    xlim=(0, 1), ylim=(0, 1.03),
)
cg_axis.legend(loc="lower left", fontsize=9)
cg_axis.grid(alpha=0.25)
pd.DataFrame(cg_pr_records).to_csv(CG_OUT / "precision_recall_points.csv", index=False)
cg_save_figure(cg_fig, "precision_recall_curves")


## Additional evaluation cell 7


In [ ]:
cg_cm = confusion_matrix(cg_y_true, cg_y_pred, labels=np.arange(CG_K))
cg_row_totals = cg_cm.sum(axis=1, keepdims=True)
cg_cm_normalized = np.divide(
    cg_cm, cg_row_totals,
    out=np.zeros_like(cg_cm, dtype=float), where=cg_row_totals != 0,
)

for cg_name, cg_values, cg_format, cg_title in [
    ("confusion_matrix_counts", cg_cm, "d", "Confusion Matrix — Counts"),
    ("confusion_matrix_normalized", cg_cm_normalized, ".2f",
     "Confusion Matrix — Row-Normalized"),
]:
    cg_fig, cg_axis = plt.subplots(figsize=(11, 9))
    ConfusionMatrixDisplay(
        confusion_matrix=cg_values, display_labels=CG_CLASSES
    ).plot(
        ax=cg_axis, cmap="Blues", values_format=cg_format,
        xticks_rotation=45, colorbar=True,
    )
    cg_axis.set_title(cg_title)
    cg_save_figure(cg_fig, cg_name)

pd.DataFrame(cg_cm, index=CG_CLASSES, columns=CG_CLASSES).to_csv(
    CG_OUT / "confusion_matrix_counts.csv"
)
pd.DataFrame(cg_cm_normalized, index=CG_CLASSES, columns=CG_CLASSES).to_csv(
    CG_OUT / "confusion_matrix_normalized.csv"
)
print("Zero-support normalized rows are displayed as zero, not measured recall.")


## Additional evaluation cell 8


In [ ]:
def cg_clean_checkpoint(source_model, name):
    inputs = tf.keras.Input(shape=(224, 224, 3), dtype="float32")
    values = source_model.get_layer("efficientnetb0")(inputs, training=False)
    values = source_model.get_layer("global_average_pooling")(values)
    values = source_model.get_layer("classification_batch_norm")(
        values, training=False
    )
    values = source_model.get_layer("classification_dropout")(
        values, training=False
    )
    values = source_model.get_layer("disease_predictions")(values)
    return tf.keras.Model(inputs, values, name=name)

cg_sample_images, _ = next(iter(test_dataset))
cg_sample_one = cg_sample_images[:1]

def cg_benchmark_model(candidate, sample, repeats=30, warmups=5):
    for _ in range(warmups):
        candidate(sample, training=False).numpy()
    values_ms = []
    for _ in range(repeats):
        started = time.perf_counter()
        candidate(sample, training=False).numpy()
        values_ms.append(1000 * (time.perf_counter() - started))
    return {
        "warmups": warmups, "repeats": repeats,
        "batch_size": int(sample.shape[0]),
        "mean_ms": float(np.mean(values_ms)),
        "median_ms": float(np.median(values_ms)),
        "p95_ms": float(np.percentile(values_ms, 95)),
        "minimum_ms": float(np.min(values_ms)),
        "maximum_ms": float(np.max(values_ms)),
        "raw_ms": values_ms,
    }

cg_comparison_rows = []
cg_comparison_timing = {}
cg_candidates = [
    ("Initial checkpoint", Path(globals().get(
        "BEST_MODEL_PATH", CG_MODEL_DIR / "best_cinnamon_leaf_model.keras"
    ))),
    ("Best fine-tuned checkpoint", Path(globals().get(
        "FINETUNE_MODEL_PATH",
        CG_MODEL_DIR / "best_cinnamon_leaf_model_finetuned.keras"
    ))),
    ("Exported final", CG_H5),
]

for cg_candidate_name, cg_candidate_path in cg_candidates:
    if not cg_candidate_path.is_file():
        print("Skipping missing checkpoint:", cg_candidate_path)
        continue
    if cg_candidate_name == "Exported final":
        cg_candidate = cg_model
        cg_candidate_true, cg_candidate_prob = cg_y_true, cg_y_prob
    else:
        cg_source_checkpoint = tf.keras.models.load_model(
            str(cg_candidate_path), compile=False
        )
        cg_candidate = cg_clean_checkpoint(
            cg_source_checkpoint, f"comparison_inference_{len(cg_comparison_rows)}"
        )
        cg_candidate_true, cg_candidate_prob, _, _ = cg_predict_dataset(cg_candidate)
        if not np.array_equal(cg_candidate_true, cg_y_true):
            raise ValueError("Test ordering changed; use the original shuffle=False dataset.")

    cg_candidate_metrics, _ = cg_classification_metrics(
        cg_candidate_true, cg_candidate_prob
    )
    cg_candidate_timing = cg_benchmark_model(cg_candidate, cg_sample_one)
    cg_comparison_timing[cg_candidate_name] = cg_candidate_timing
    cg_comparison_rows.append({
        "checkpoint": cg_candidate_name,
        "accuracy": cg_candidate_metrics["accuracy"],
        "precision_macro": cg_candidate_metrics["precision_macro"],
        "recall_macro": cg_candidate_metrics["recall_macro"],
        "f1_macro": cg_candidate_metrics["f1_macro"],
        "roc_auc_macro": cg_candidate_metrics["roc_auc_macro_all_classes"],
        "median_single_image_ms": cg_candidate_timing["median_ms"],
        "p95_single_image_ms": cg_candidate_timing["p95_ms"],
        "parameters": int(cg_candidate.count_params()),
        "artifact_bytes": cg_candidate_path.stat().st_size,
    })
    if cg_candidate_name != "Exported final":
        del cg_candidate, cg_source_checkpoint

cg_comparison = pd.DataFrame(cg_comparison_rows)
cg_comparison.to_csv(CG_OUT / "checkpoint_comparison.csv", index=False)
cg_save_json("checkpoint_timing.json", cg_comparison_timing)
display(cg_comparison)

cg_fig, cg_axis = plt.subplots(figsize=(12, 6))
cg_comparison.set_index("checkpoint")[[
    "accuracy", "precision_macro", "recall_macro", "f1_macro"
]].plot.bar(ax=cg_axis)
cg_axis.set(
    ylabel="Score (0–1)", ylim=(0, 1.08),
    title="Model Comparison — Same Architecture, Different Checkpoints",
)
cg_axis.tick_params(axis="x", rotation=15)
cg_axis.legend(loc="lower right")
cg_save_figure(cg_fig, "model_comparison")

cg_fig, cg_axis = plt.subplots(figsize=(10, 6))
cg_comparison.set_index("checkpoint")[[
    "median_single_image_ms", "p95_single_image_ms"
]].plot.bar(ax=cg_axis)
cg_axis.set(
    ylabel="Milliseconds", title="Performance Comparison — Warm Single-Image Execution"
)
cg_axis.tick_params(axis="x", rotation=15)
cg_save_figure(cg_fig, "performance_comparison")
print("File sizes may include different serialization/training states; not just weights.")


## Additional evaluation cell 9


In [ ]:
# Categorical view: inspect matches without treating class IDs as measurements.
cg_rows_to_show = min(50, len(cg_predictions))
cg_label_table = cg_predictions.iloc[:cg_rows_to_show][[
    "evaluation_row", "actual_class", "predicted_class", "correct"
]]
display(cg_label_table)
cg_fig, cg_axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
cg_label_map = plt.get_cmap("tab10", CG_K)
for cg_axis, cg_values, cg_title in [
    (cg_axes[0], cg_y_true[:cg_rows_to_show], "Actual class"),
    (cg_axes[1], cg_y_pred[:cg_rows_to_show], "Predicted class"),
]:
    cg_axis.imshow(
        cg_values[None, :], aspect="auto", cmap=cg_label_map,
        vmin=-0.5, vmax=CG_K-0.5, interpolation="nearest",
    )
    cg_axis.set_yticks([])
    cg_axis.set_ylabel(cg_title)
cg_axes[1].set_xlabel("Evaluation row — first 50 examples at most")
cg_fig.suptitle("Actual vs Predicted Disease Labels — Categorical Display")
cg_handles = [
    plt.Line2D([0], [0], marker="s", linestyle="", color=cg_label_map(i),
               label=name, markersize=10)
    for i, name in enumerate(CG_CLASSES)
]
cg_axes[0].legend(handles=cg_handles, bbox_to_anchor=(1.02, 1), loc="upper left")
cg_save_figure(cg_fig, "actual_vs_predicted_class_labels")

CG_REGRESSION_CSV = None  # Example: "/content/drive/.../bark_measurements.csv"
CG_REGRESSION_UNIT = "mm"  # Change to the actual target's unit.

if CG_REGRESSION_CSV is None:
    print("RMSE, MAE, MAPE, R²: not applicable to unordered disease class IDs.")
    print("Set CG_REGRESSION_CSV to a measured continuous-target table if available.")
else:
    cg_regression = pd.read_csv(CG_REGRESSION_CSV)
    if not {"actual", "predicted"}.issubset(cg_regression.columns):
        raise ValueError("Regression CSV requires actual and predicted columns.")
    cg_actual = pd.to_numeric(cg_regression["actual"], errors="raise").to_numpy()
    cg_estimate = pd.to_numeric(cg_regression["predicted"], errors="raise").to_numpy()
    if len(cg_actual) < 2 or not np.all(np.isfinite([cg_actual, cg_estimate])):
        raise ValueError("Need at least two finite paired regression records.")
    cg_regression_metrics = {
        "samples": len(cg_actual), "unit": CG_REGRESSION_UNIT,
        "RMSE": float(np.sqrt(mean_squared_error(cg_actual, cg_estimate))),
        "MAE": float(mean_absolute_error(cg_actual, cg_estimate)),
        "MAPE_percent": (
            float(100 * mean_absolute_percentage_error(cg_actual, cg_estimate))
            if np.all(cg_actual != 0) else None
        ),
        "R2": (
            float(r2_score(cg_actual, cg_estimate))
            if np.var(cg_actual) > 0 else None
        ),
    }
    if cg_regression_metrics["MAPE_percent"] is None:
        print("MAPE undefined: zero actual targets. It is left unavailable.")
    if cg_regression_metrics["R2"] is None:
        print("R² undefined: constant actual target. It is left unavailable.")
    cg_save_json("regression_metrics.json", cg_regression_metrics)
    cg_regression.to_csv(CG_OUT / "regression_pairs.csv", index=False)
    display(pd.DataFrame([cg_regression_metrics]).T)
    cg_fig, cg_axis = plt.subplots(figsize=(7, 6))
    cg_axis.scatter(cg_actual, cg_estimate, alpha=0.6)
    cg_low = min(cg_actual.min(), cg_estimate.min())
    cg_high = max(cg_actual.max(), cg_estimate.max())
    if cg_low == cg_high:
        cg_low, cg_high = cg_low - 0.5, cg_high + 0.5
    cg_axis.plot([cg_low, cg_high], [cg_low, cg_high], "k--", label="Perfect agreement")
    cg_axis.set(
        xlabel=f"Actual ({CG_REGRESSION_UNIT})",
        ylabel=f"Predicted ({CG_REGRESSION_UNIT})",
        title="Actual vs Predicted — Continuous Target",
    )
    cg_axis.legend()
    cg_axis.grid(alpha=0.25)
    cg_save_figure(cg_fig, "actual_vs_predicted_regression")


## Additional evaluation cell 10


In [ ]:
cg_final_single_timing = cg_benchmark_model(cg_model, cg_sample_one, repeats=50)
cg_final_batch_timing = cg_benchmark_model(cg_model, cg_sample_images, repeats=30)
cg_timing_report = {
    "single_image": cg_final_single_timing,
    "batch": cg_final_batch_timing,
    "held_out_model_call_seconds": cg_eval_model_seconds,
    "held_out_loop_seconds": cg_eval_loop_seconds,
    "scope": "Local warmed model calls; no HTTP or external services",
    "training_execution_seconds": None,
    "training_time_note": "The original fit calls were not timed; duration cannot be recovered afterward.",
}
cg_save_json("execution_timing.json", cg_timing_report)

cg_timing_table = pd.DataFrame([
    {"workload": name, **{k: v for k, v in timing.items() if k != "raw_ms"}}
    for name, timing in [
        ("Single image", cg_final_single_timing),
        ("Dataset batch", cg_final_batch_timing),
    ]
])
display(cg_timing_table)
cg_timing_table.to_csv(CG_OUT / "execution_timing_summary.csv", index=False)
cg_fig, cg_axis = plt.subplots(figsize=(10, 5))
cg_timing_table.set_index("workload")[["mean_ms", "median_ms", "p95_ms"]].plot.bar(
    ax=cg_axis
)
cg_axis.set(ylabel="Milliseconds per call", title="Final Model — Warm Execution Time")
cg_axis.tick_params(axis="x", rotation=0)
cg_save_figure(cg_fig, "execution_time")


## Additional evaluation cell 11


In [ ]:
CG_RUN_HTTP_BENCHMARK = False
CG_HTTP_URL = None  # Example: "https://your-test-backend/disease-predict/"
CG_HTTP_IMAGE = None  # Example: "/content/drive/.../test-leaf.jpg"
CG_HTTP_REPEATS = 10

if not CG_RUN_HTTP_BENCHMARK:
    print("HTTP benchmark skipped. Local inference timing is available in Cell 10.")
else:
    import getpass
    import mimetypes
    import requests

    if not CG_HTTP_URL or not CG_HTTP_IMAGE:
        raise ValueError("Set CG_HTTP_URL and CG_HTTP_IMAGE before enabling.")
    cg_image_path = Path(CG_HTTP_IMAGE)
    cg_upload_bytes = cg_image_path.read_bytes()
    cg_mime = mimetypes.guess_type(cg_image_path.name)[0] or "image/jpeg"
    cg_token = getpass.getpass("Firebase ID token for the test account: ").strip()
    if not cg_token:
        raise ValueError("A Firebase ID token is required.")
    cg_http_records = []
    with requests.Session() as cg_session:
        cg_session.headers.update({"Authorization": f"Bearer {cg_token}"})
        for cg_request_index in range(CG_HTTP_REPEATS):
            cg_started = time.perf_counter()
            try:
                cg_response = cg_session.post(
                    CG_HTTP_URL,
                    files={"file": (cg_image_path.name, cg_upload_bytes, cg_mime)},
                    timeout=(10, 120),
                )
                # requests reads the body before returning with stream=False.
                cg_duration_ms = 1000 * (time.perf_counter() - cg_started)
                cg_status = int(cg_response.status_code)
                cg_success = 200 <= cg_status < 300
                cg_error = None
            except requests.RequestException as cg_exception:
                cg_duration_ms = 1000 * (time.perf_counter() - cg_started)
                cg_status, cg_success = None, False
                cg_error = type(cg_exception).__name__
            cg_http_records.append({
                "request": cg_request_index + 1, "response_ms": cg_duration_ms,
                "http_status": cg_status, "success": cg_success, "error": cg_error,
            })
    del cg_token
    cg_http_table = pd.DataFrame(cg_http_records)
    cg_http_table.to_csv(CG_OUT / "http_response_times.csv", index=False)
    cg_ok_times = cg_http_table.loc[cg_http_table.success, "response_ms"].to_numpy()
    cg_http_summary = {
        "attempted": CG_HTTP_REPEATS,
        "successful": int(cg_http_table.success.sum()),
        "failed": int((~cg_http_table.success).sum()),
        "upload_bytes": len(cg_upload_bytes),
        "mean_success_ms": float(np.mean(cg_ok_times)) if len(cg_ok_times) else None,
        "median_success_ms": float(np.median(cg_ok_times)) if len(cg_ok_times) else None,
        "p95_success_ms": float(np.percentile(cg_ok_times, 95)) if len(cg_ok_times) else None,
        "scope": "Sequential HTTPS calls from Colab; includes transport and enabled backend integrations",
        "warmup_requests": 0,
    }
    cg_save_json("http_response_summary.json", cg_http_summary)
    display(cg_http_table)
    display(pd.DataFrame([cg_http_summary]).T)
    cg_fig, cg_axis = plt.subplots(figsize=(10, 5))
    cg_axis.bar(
        cg_http_table.request, cg_http_table.response_ms,
        color=["tab:blue" if x else "tab:red" for x in cg_http_table.success],
    )
    cg_axis.set(
        xlabel="Sequential request", ylabel="Response duration (ms)",
        title="HTTP Response Times — Blue: Success, Red: Failure",
    )
    cg_save_figure(cg_fig, "http_response_time")


## Additional evaluation cell 12


In [ ]:
cg_manifest = sorted(
    str(path.relative_to(CG_OUT)) for path in CG_OUT.rglob("*") if path.is_file()
)
cg_save_json("manifest.json", {
    "run_id_utc": CG_RUN_ID,
    "files_before_manifest": cg_manifest,
    "notes": [
        "Original model and training files are unchanged.",
        "History curves require retained training history.",
        "Continuous regression metrics require a separate paired target dataset.",
        "Local model timing and real HTTP timing have different scopes.",
    ],
})
CG_ZIP = CG_OUT.parent / f"{CG_OUT.name}.zip"
with zipfile.ZipFile(CG_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as cg_archive:
    for cg_file in sorted(CG_OUT.rglob("*")):
        if cg_file.is_file():
            cg_archive.write(cg_file, arcname=cg_file.relative_to(CG_OUT))
print("Saved report folder:", CG_OUT)
print("Saved ZIP:", CG_ZIP)

# Optional browser download in Colab:
# from google.colab import files
# files.download(str(CG_ZIP))
